# cpt-anywidget in Jupyter

The widgets are plain `anywidget.AnyWidget`s, so they should work in any
Jupyter front end without the marimo-specific wiring. This notebook mirrors
`bro-xml-explore.py`'s data loading with static values in place of the marimo
UI controls.

Run from the repo root (JupyterLab lives in the `jupyter` dependency
group — it must be in the *project* env, not a `--with` overlay, or
JupyterLab won't find the widget labextensions and cells show a plain
`repr` instead of the widget):

```sh
uv run --group jupyter jupyter lab notebooks/broxml-explore.ipynb
```

Things to check per widget:

- **CPTViewer** — hover crosshair, wheel/drag zoom on the vertical axis,
  and the edit column: drag boundaries, double-click to split, option-click
  to merge, click a layer for the soil-class pie. Edits must sync back
  (re-run the `editedLayers` cell after a drag).
- **ProfileViewer** — true-scale ↔ equal spacing toggle, strip click
  syncing `selected` back.
- **BoreholeViewer** — soil-composition bands + hatches, same zoom/hover.
- **Teardown** — re-running a viewer cell replaces the view; no errors in
  the browser console (checks the render dispose/AbortSignal path).

In [ ]:
from pathlib import Path

import pandas as pd

from cpt_anywidget import (
    BoreholeViewer,
    CPTViewer,
    ProfileViewer,
    chainage,
    layers_from_bhrgt,
)
from brodata.cpt import ConePenetrationTest
from brodata.bhr import GeotechnicalBoreholeResearch

# works whether the kernel starts in the repo root or in notebooks/
ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "examples" / "broxml-cpt").is_dir()
)

In [ ]:
cpt_files = sorted((ROOT / "examples" / "broxml-cpt").glob("*.xml"))
cpt = ConePenetrationTest(str(cpt_files[1]))
print(cpt.broId, "—", cpt.description or cpt.deliveryContext)
print(f"final depth {cpt.finalDepth} m, surface at {cpt.offset} m NAP")

CPT000000179849 — Bandentruck
final depth 20.062 m, surface at 1.560 m NAP


In [ ]:
# tidy columns straight from brodata: pick the known channels, rename the
# one awkward BRO name, coerce numeric strings. NaN → None and list
# conversion happen in the CPTViewer facade, so no hand-scrubbing here
df = cpt.conePenetrationTest.dropna(axis=1, how="all").sort_index()
cpt_data = (
    df.rename(columns={"inclinationResultant": "inclination"})
    .reindex(
        columns=[
            "depth",
            "coneResistance",
            "localFriction",
            "frictionRatio",
            "inclination",
            "porePressureU1",
            "porePressureU2",
        ]
    )
    .apply(pd.to_numeric, errors="coerce")
    .dropna(axis="columns", how="all")
)

# vertical coordinate in m NAP: surface elevation minus depth below surface
cpt_data["nap"] = float(cpt.offset) - cpt_data["depth"]

channels = [
    c
    for c in ("coneResistance", "localFriction", "frictionRatio", "porePressureU2")
    if c in cpt_data
]
cpt_data.head()

,depth,coneResistance,localFriction,frictionRatio,inclination,porePressureU2,nap
penetrationLength,,,,,,,
1.18,1.18,10.44,0.083,0.8,0,0.0,0.38
1.20,1.20,9.73,0.081,0.9,0,0.0,0.36
1.22,1.22,8.78,0.078,0.9,0,0.0,0.34
1.24,1.24,7.99,0.070,0.8,0,0.0,0.32
1.26,1.26,7.72,0.062,0.8,0,0.0,0.30


## CPTViewer — depth mode, interpretations + editable column

Layers are class-keyed, so fills come from the default `soil_classes`
palette (also feeds the pie menu). The edit column is seeded from the
Robertson interpretation directly — no marimo seed radio here.

In [ ]:
final = float(cpt.finalDepth)
predrilled = float(cpt.predrilledDepth)

# dummy soil interpretations, boundaries in m below surface
interpretations = [
    {
        "label": "Robertson",
        "layers": [
            {"top": predrilled, "bottom": 2.4, "class": "sand"},
            {"top": 2.4, "bottom": 6.8, "class": "clay"},
            {"top": 6.8, "bottom": 9.5, "class": "peat"},
            {"top": 9.5, "bottom": final, "class": "sand"},
        ],
    },
    {
        "label": "CPT-Core-A",
        "layers": [
            {"top": predrilled, "bottom": 2.1, "class": "sand"},
            {"top": 2.1, "bottom": 6.2, "class": "clay"},
            {"top": 6.2, "bottom": 10.3, "class": "peat"},
            {"top": 10.3, "bottom": final, "class": "sand"},
        ],
    },
]

# groundwater level (m below surface), shared by the GWL annotation and
# the hydrostatic overlay (0.00981 MPa per m of water column); the
# overlay only renders while porePressureU2 is plotted
gwl = 2.5
hydrostatic = {
    "channel": "porePressureU2",
    "points": [[0, gwl], [0.00981 * (final - gwl), final]],
    "color": "#4269d0",
    "dash": "4,3",
}

viewer = CPTViewer(
    cpt_data,
    vertical="depth",
    channels=channels,
    interpretations=interpretations,
    editedLayers=[dict(l) for l in interpretations[0]["layers"]],
    overlays=[hydrostatic],
    annotations=[
        {"at": gwl, "label": "GWL", "color": "#4269d0", "position": "left"},
        {
            "at": predrilled,
            "label": "Voorgeboorde diepte",
            "color": "#000000",
            "position": "right",
        },
    ],
)
viewer

In [ ]:
# drag a boundary (or split/merge) in the "Edited Interpr." column above,
# then re-run this cell: edits sync back over the comm into the trait
pd.DataFrame(viewer.editedLayers)

,top,bottom,class
0,1.18,2.400,sand
1,2.40,6.800,clay
2,6.80,9.500,peat
3,9.50,20.062,sand


## NAP mode + borehole column

The borehole column only makes sense on the NAP axis (the only datum a CPT
and a nearby borehole genuinely share), so this second viewer switches
`vertical="nap"` and converts the layer boundaries and annotations at the
widget boundary.

In [ ]:
bhr_files = sorted((ROOT / "examples" / "broxml-bhr-gt").glob("*.xml"))
gt_borehole = GeotechnicalBoreholeResearch(str(bhr_files[0]))


def nap(depth_below_surface):
    return float(cpt.offset) - depth_below_surface


CPTViewer(
    cpt_data,
    vertical="nap",
    channels=channels,
    interpretations=[
        {
            "label": col["label"],
            "layers": [
                {**l, "top": nap(l["top"]), "bottom": nap(l["bottom"])}
                for l in col["layers"]
            ],
        }
        for col in interpretations
    ],
    borehole={
        "label": gt_borehole.broId,
        "layers": layers_from_bhrgt(gt_borehole, "nap"),
    },
    annotations=[
        {"at": nap(gwl), "label": "GWL", "color": "#4269d0", "position": "left"}
    ],
)

Tag roughness  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag roughness  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag determinationProcedure  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag determinationMethod  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag fractionDistribution  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag dispersionMethod  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag removedMaterial  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag equivalentMassDeterminationMethod  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag equivalentMass  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag basicParticleSizeDistribution  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag organicMatterContentDetermination  not supported in GeotechnicalBoreholeResearch BHR000000377186
Tag organicMatterContentDetermina

## ProfileViewer — length profile

The two sample CPTs sit ~26 km apart, so true-scale chainage is an honest
but extreme axis — use the toolbar toggle to switch to equal spacing.
Clicking a strip syncs its name back via `selected`.

In [ ]:
# every sample CPT in one tidy long frame — name + nap + qc per row, the
# ProfileViewer facade groups rows by the name column — plus chainage
# positions from the delivered RD coordinates and each CPT's surface
# level for the maaiveld overlay
cpts = [ConePenetrationTest(str(p)) for p in cpt_files]
profile_data = pd.concat(
    [
        pd.DataFrame(
            {
                "name": c.broId,
                "nap": float(c.offset) - d["depth"],
                "coneResistance": d["coneResistance"],
            }
        )
        for c in cpts
        for d in [c.conePenetrationTest.apply(pd.to_numeric, errors="coerce")]
    ],
    ignore_index=True,
)
positions = chainage(
    {c.broId: (c.deliveredLocation.x, c.deliveredLocation.y) for c in cpts}
)
surface_levels = {c.broId: float(c.offset) for c in cpts}

profile = ProfileViewer(
    profile_data,
    positions=positions,
    channel="coneResistance",
    overlays=[
        {"levels": surface_levels, "label": "maaiveld", "color": "#8a6642"}
    ],
    height=420,
)
profile

In [ ]:
# click a strip above, then re-run: its name syncs back ("" = none;
# clicking the selected strip again deselects)
profile.selected

''

## BoreholeViewer — standalone borehole log

In [ ]:
BoreholeViewer(layers=layers_from_bhrgt(gt_borehole, "depth"))